In [ ]:
# #################################################################
# LASO (Local Adaptive Synthetic Oversampling): A Python implementation for multiclass imbalanced learning using local neighborhood and PCA-based synthetic sample generation.
# Prepared on 02-Feb-2026 (by I Made Putrama)
# #################################################################
import os, glob
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
import math
from datetime import datetime
import time

import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter

from imblearn.under_sampling import RandomUnderSampler, NearMiss, TomekLinks, EditedNearestNeighbours
from imblearn.combine import SMOTEENN, SMOTETomek
from imblearn.over_sampling import RandomOverSampler, SMOTE, SMOTENC, ADASYN, BorderlineSMOTE, SVMSMOTE, KMeansSMOTE
from imblearn.ensemble import EasyEnsembleClassifier, RUSBoostClassifier, BalancedBaggingClassifier, BalancedRandomForestClassifier

from smotecdnn import EditedCDNN, SMOTECDNN

from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

DATA_DIR = "data.all"
OUTPUT_DIR = "results.all"
DELIM = ","
LABEL_COLUMNS = ['target','label','class','attack','quality','y']
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# The following is to handle parameter related errors on the EditedCDNN & SMOTECDNN
try:
    if not hasattr(EditedCDNN, "_parameter_constraints"):
        EditedCDNN._parameter_constraints = {}
except Exception:
    pass

try:
    if not hasattr(SMOTECDNN, "_parameter_constraints"):
        SMOTECDNN._parameter_constraints = {}
except Exception:
    pass

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Local Adaptive Synthetic Oversampling (LASO) method to balance multiclass imbalanced datasets.
def laso(X, y, target_per_class=None, random_state=42,
            k_neighbors=5, var_explained=0.95, max_components=None,
            max_retries=50, perturb_scale=1e-6):
    """
    X : ndarray, shape (n_samples, n_features) Feature matrix.
    y : ndarray, shape (n_samples,) Class labels (can be numeric or strings). 
    target_per_class : int or None. Desired number of samples per class. If None, set to the majority class count.
    random_state : int. Random seed.
    k_neighbors : int. Number of neighbors to use for local PCA (neighbors + the seed point).
    var_explained : float in (0,1]. Fraction of variance to keep in PCA (local dimensionality).
    max_components : int or None. Upper bound on number of PCA components. If None, no explicit cap.
    max_retries : int. Number of attempts to find an acceptable candidate before fallback.
    perturb_scale : float. Scale of tiny Gaussian noise used when a class has only one sample.
    
    Returns
    -------
    X_res : ndarray. Resampled feature matrix.
    y_res : ndarray. Resampled labels (same dtype as input y).
    """

    np.random.seed(random_state)
    rng = np.random.RandomState(random_state)

    classes = np.unique(y)
    # Determine target. Use majority class count if not provided
    counts = {c: int(np.sum(y == c)) for c in classes}
    if target_per_class is None:
        target_per_class = int(max(counts.values()))

    # Build class-wise arrays
    class_points = {c: X[y == c].copy() for c in classes}
    # Update centroids incrementally when adding samples
    def compute_centroid(arr):
        return np.mean(arr, axis=0) if len(arr) > 0 else None

    def incremental_centroid_add(mu_old, n_old, x_new):
        if n_old == 0 or mu_old is None:
            return x_new.copy()
        return (n_old * mu_old + x_new) / (n_old + 1)

    # For each class
    for c in classes:
        pts = class_points[c]
        n_c = len(pts)
        if n_c >= target_per_class:
            continue  # nothing to do

        # prepare neighbor search on current class points
        # If class has >= 2 points, use local PCA interpolation
        build_tree = True

        # current centroid
        mu = compute_centroid(pts)

        # Loop until taret is reached
        attempts = 0
        while len(class_points[c]) < target_per_class:
            pts = class_points[c]
            n_c = len(pts)
            mu = compute_centroid(pts)
            
            # if < 2 points -> perturb base sample
            if n_c == 0:
                # nothing to do, break
                break
            if n_c == 1:
                # simple perturbation using single sample
                base = pts[0]
                x_new = base + rng.normal(scale=perturb_scale, size=base.shape)
                class_points[c] = np.vstack([class_points[c], x_new])
                attempts = 0
                continue

            # ensure local neigbors exist
            k_local = min(k_neighbors, n_c - 1)

            # rebuild neighbor index
            nbrs = NearestNeighbors(n_neighbors=k_local + 1, algorithm='auto').fit(pts)
            
            # pick a random seed index from current points
            seed_idx = rng.randint(0, n_c)            
            distances, indices = nbrs.kneighbors(pts[seed_idx].reshape(1, -1), return_distance=True)
            
            # indices shape (1, k_local+1)
            local_idx = indices[0]  # includes seed at position 0
            local_pts = pts[local_idx]  # shape (k_local+1, d)

            # center local patch
            local_mean = np.mean(local_pts, axis=0)
            Xc = local_pts - local_mean

            # compute local PCA. Use sklearn PCA with svd_solver='full'
            n_components_possible = min(Xc.shape[0], Xc.shape[1])
            if max_components is not None:
                n_components_possible = min(n_components_possible, max_components)
            pca = PCA(n_components=n_components_possible, svd_solver='full', random_state=random_state)
            try:
                pca.fit(Xc)
            except Exception:
                # simple linear interpolation between two neighbors
                if Xc.shape[0] >= 2:
                    ia, ib = rng.choice(Xc.shape[0], size=2, replace=False)
                    za = Xc[ia]
                    zb = Xc[ib]
                    lam = rng.rand()
                    z_new = (1 - lam) * za + lam * zb
                    x_new = local_mean + z_new
                    class_points[c] = np.vstack([class_points[c], x_new])
                else:
                    # perturb one point
                    base = pts[seed_idx]
                    x_new = base + rng.normal(scale=perturb_scale, size=base.shape)
                    class_points[c] = np.vstack([class_points[c], x_new])
                continue

            # determine number of components to keep based on var_explained
            cumvar = np.cumsum(pca.explained_variance_ratio_)
            m = np.searchsorted(cumvar, var_explained) + 1
            if m < 1:
                m = 1
            m = min(m, pca.components_.shape[0])
            if max_components is not None:
                m = min(m, max_components)

            # project local points into m-dim PCA space
            V = pca.components_[:m]  # shape (m, d)
            Z = Xc.dot(V.T)  # (k_local+1, m)

            # sample two local coordinates and interpolate in PCA space
            retries = 0
            accepted = False
            last_candidate = None
            while retries < max_retries and not accepted:
                # pick two distinct indices among local points
                if Z.shape[0] >= 2:
                    ia, ib = rng.choice(Z.shape[0], size=2, replace=False)
                    za = Z[ia]
                    zb = Z[ib]
                    lam = rng.rand()
                    z_new = (1 - lam) * za + lam * zb  # in PCA coordinates
                else:
                    # perform small perturbation in PCA space
                    za = Z[0]
                    z_new = za + rng.normal(scale=perturb_scale, size=za.shape)

                # map back to original feature space
                x_new = local_mean + z_new.dot(V)  # (d,)
                last_candidate = x_new
                # accept the generated point (no radius consideration)
                class_points[c] = np.vstack([class_points[c], x_new])
                accepted = True
                break

                retries += 1

            if not accepted:
                # project last_candidate near centroid or perturb centroid
                if last_candidate is not None:
                    class_points[c] = np.vstack([class_points[c], last_candidate])
                else:
                    base = pts[seed_idx]
                    x_new = base + rng.normal(scale=perturb_scale, size=base.shape)
                    class_points[c] = np.vstack([class_points[c], x_new])

    # get final arrays
    X_res_list = []
    y_res_list = []
    for c in classes:
        pts = class_points[c]
        for p in pts:
            X_res_list.append(p)
            y_res_list.append(c)

    X_res = np.vstack(X_res_list) if len(X_res_list) > 0 else np.empty((0, X.shape[1]))
    y_res = np.array(y_res_list, dtype=y.dtype)
    return X_res, y_res

# Local Adaptive Synthetic Oversampling (LASO) method to balance multiclass imbalanced datasets with .
def laso_r(X, y, target_per_class=None, random_state=42,
           k_neighbors=5, var_explained=0.95, max_components=None,
           max_retries=50, perturb_scale=1e-6, delta_r=1e-12):
    """
    Local-PCA based oversampling **with radius check**.
    - For each class with count < target_per_class (default = majority class size),
      generate synthetic samples by performing interpolation in a local PCA
      subspace computed from a sample and its k nearest same-class neighbors.
    - **Only accept** synthetic samples that lie within the class' non-overlapping
      radius (computed from class centroid to nearest other-class point minus small delta).
    - Recompute centroid (and radius) after each accepted sample.
    - No discarding of existing points.
    """
    np.random.seed(random_state)
    rng = np.random.RandomState(random_state)

    classes = np.unique(y)
    # Determine target. Use majority class count if not provided
    counts = {c: int(np.sum(y == c)) for c in classes}
    if target_per_class is None:
        target_per_class = int(max(counts.values()))

    # Build class-wise arrays
    class_points = {c: X[y == c].copy() for c in classes}

    # helper method to compute centroid
    def compute_centroid(arr):
        return np.mean(arr, axis=0) if len(arr) > 0 else None

    # helper method to compute centroid
    def incremental_centroid_add(mu_old, n_old, x_new):
        if n_old == 0 or mu_old is None:
            return x_new.copy()
        return (n_old * mu_old + x_new) / (n_old + 1)

    # compute initial centroids
    centroids = {c: compute_centroid(class_points[c]) for c in classes}

    # compute initial radii using compute_radius if available,
    # else compute minimum distance to other-class centroids/points
    radii = {}
    for c in classes:
        other_pts = np.vstack([class_points[o] for o in classes if o != c and len(class_points[o]) > 0]) \
            if any(len(class_points[o]) > 0 for o in classes if o != c) else np.empty((0, X.shape[1]))
        mu = centroids[c]
        if mu is None:
            radii[c] = 0.0
        else:
            try:
                r = compute_radius(mu, class_points[c], other_pts)
            except Exception:
                if other_pts.shape[0] == 0:
                    dists = np.linalg.norm(class_points[c] - mu, axis=1) if len(class_points[c]) > 0 else np.array([0.0])
                    r = float(dists.max()) if len(dists) > 0 else 0.0
                else:
                    dists = np.linalg.norm(other_pts - mu, axis=1)
                    r = float(dists.min() - delta_r)
            if np.isnan(r) or r < 0:
                r = 0.0
            radii[c] = r

    # For each class that needs augmentation
    for c in classes:
        pts = class_points[c]
        n_c = len(pts)
        if n_c >= target_per_class:
            continue  # nothing to do

        # loop until reach target
        while len(class_points[c]) < target_per_class:
            pts = class_points[c]
            n_c = len(pts)
            mu = compute_centroid(pts)
            # recompute radius with current centroid and other-class points
            other_pts = np.vstack([class_points[o] for o in classes if o != c and len(class_points[o]) > 0]) \
                if any(len(class_points[o]) > 0 for o in classes if o != c) else np.empty((0, X.shape[1]))
            if mu is None:
                break
            try:
                r = compute_radius(mu, pts, other_pts)
            except Exception:
                if other_pts.shape[0] == 0:
                    dtmp = np.linalg.norm(pts - mu, axis=1) if len(pts) > 0 else np.array([0.0])
                    r = float(dtmp.max()) if len(dtmp) > 0 else 0.0
                else:
                    r = float(np.linalg.norm(other_pts - mu, axis=1).min() - delta_r)
            if np.isnan(r) or r < 0:
                r = 0.0
            radii[c] = r

            # if fewer than 1 seed, break
            if n_c == 0:
                break
            if n_c == 1:
                # small perturbation only if within radius
                base = pts[0]
                x_new = base + rng.normal(scale=perturb_scale, size=base.shape)
                if np.linalg.norm(x_new - mu) <= r + 1e-12:
                    class_points[c] = np.vstack([class_points[c], x_new])
                else:
                    # otherwise, retry small perturbations up to max_retries
                    retr = 0
                    accepted = False
                    while retr < max_retries and not accepted:
                        x_new = base + rng.normal(scale=perturb_scale, size=base.shape)
                        if np.linalg.norm(x_new - mu) <= r + 1e-12:
                            class_points[c] = np.vstack([class_points[c], x_new])
                            accepted = True
                        retr += 1
                    if not accepted:
                        # perform small perturbation if r>0
                        if r > 0:
                            v = base - mu
                            if np.linalg.norm(v) == 0:
                                # pick tiny random direction
                                v = rng.normal(size=base.shape)
                            x_proj = mu + r * v / np.linalg.norm(v)
                            class_points[c] = np.vstack([class_points[c], x_proj])
                        else:
                            # accept small noise around base (even if outside)
                            x_small = base + rng.normal(scale=perturb_scale, size=base.shape)
                            class_points[c] = np.vstack([class_points[c], x_small])
                # update centroid incrementally
                centroids[c] = compute_centroid(class_points[c])
                continue  # continue loop

            # Build local neighborhood
            k_local = min(k_neighbors, n_c - 1)
            nbrs = NearestNeighbors(n_neighbors=k_local + 1, algorithm='auto').fit(pts)
            seed_idx = rng.randint(0, n_c)
            distances, indices = nbrs.kneighbors(pts[seed_idx].reshape(1, -1), return_distance=True)
            local_idx = indices[0]
            local_pts = pts[local_idx]

            # center local patch and PCA
            local_mean = np.mean(local_pts, axis=0)
            Xc = local_pts - local_mean
            n_components_possible = min(Xc.shape[0], Xc.shape[1])
            if max_components is not None:
                n_components_possible = min(n_components_possible, max_components)
            pca = PCA(n_components=n_components_possible, svd_solver='full', random_state=random_state)
            try:
                pca.fit(Xc)
            except Exception:
                # perform linear interpolation between two local points
                if Xc.shape[0] >= 2:
                    ia, ib = rng.choice(Xc.shape[0], size=2, replace=False)
                    za = Xc[ia]; zb = Xc[ib]
                    lam = rng.rand()
                    z_new = (1 - lam) * za + lam * zb
                    x_candidate = local_mean + z_new
                else:
                    base = pts[seed_idx]
                    x_candidate = base + rng.normal(scale=perturb_scale, size=base.shape)
                # check radius and accept if within it
                if np.linalg.norm(x_candidate - mu) <= r + 1e-12:
                    class_points[c] = np.vstack([class_points[c], x_candidate])
                    centroids[c] = compute_centroid(class_points[c])
                else:
                    # try limited retries
                    retr = 0; accepted = False
                    while retr < max_retries and not accepted:
                        if Xc.shape[0] >= 2:
                            ia, ib = rng.choice(Xc.shape[0], size=2, replace=False)
                            za = Xc[ia]; zb = Xc[ib]
                            lam = rng.rand()
                            z_new = (1 - lam) * za + lam * zb
                            x_candidate = local_mean + z_new
                        else:
                            x_candidate = base + rng.normal(scale=perturb_scale, size=base.shape)
                        if np.linalg.norm(x_candidate - mu) <= r + 1e-12:
                            class_points[c] = np.vstack([class_points[c], x_candidate])
                            centroids[c] = compute_centroid(class_points[c])
                            accepted = True
                        retr += 1
                    if not accepted:
                        # perform projection if possible
                        if r > 0:
                            v = (x_candidate - mu)
                            if np.linalg.norm(v) > 0:
                                x_proj = mu + r * v / np.linalg.norm(v)
                                class_points[c] = np.vstack([class_points[c], x_proj])
                                centroids[c] = compute_centroid(class_points[c])
                continue  # proceed to next iteration

            # choose local PCA dimension m
            cumvar = np.cumsum(pca.explained_variance_ratio_)
            m = np.searchsorted(cumvar, var_explained) + 1
            if m < 1:
                m = 1
            m = min(m, pca.components_.shape[0])
            if max_components is not None:
                m = min(m, max_components)

            V = pca.components_[:m]  # (m, d)
            Z = Xc.dot(V.T)         # (k_local+1, m)

            # attempt sampling within PCA space and accept if within radius
            retries = 0
            accepted = False
            last_candidate = None
            while retries < max_retries and not accepted:
                if Z.shape[0] >= 2:
                    ia, ib = rng.choice(Z.shape[0], size=2, replace=False)
                    za = Z[ia]; zb = Z[ib]
                    lam = rng.rand()
                    z_new = (1 - lam) * za + lam * zb
                else:
                    za = Z[0]
                    z_new = za + rng.normal(scale=perturb_scale, size=za.shape)
                x_candidate = local_mean + z_new.dot(V)
                last_candidate = x_candidate
                if np.linalg.norm(x_candidate - mu) <= r + 1e-12:
                    # accept
                    class_points[c] = np.vstack([class_points[c], x_candidate])
                    centroids[c] = compute_centroid(class_points[c])
                    accepted = True
                    break
                retries += 1

            if not accepted:
                # perform projection last_candidate onto radius boundary if available
                if last_candidate is not None and r > 0:
                    v = last_candidate - mu
                    if np.linalg.norm(v) > 0:
                        x_proj = mu + r * v / np.linalg.norm(v)
                        class_points[c] = np.vstack([class_points[c], x_proj])
                        centroids[c] = compute_centroid(class_points[c])
                    else:
                        # tiny noise around mu
                        x_small = mu + rng.normal(scale=perturb_scale, size=mu.shape)
                        class_points[c] = np.vstack([class_points[c], x_small])
                        centroids[c] = compute_centroid(class_points[c])
                else:
                    # last resort: tiny noise around mu
                    x_small = mu + rng.normal(scale=perturb_scale, size=mu.shape)
                    class_points[c] = np.vstack([class_points[c], x_small])
                    centroids[c] = compute_centroid(class_points[c])
            # end of per-sample loop; will re-check len and continue

    # collect final arrays
    X_res_list = []
    y_res_list = []
    for c in classes:
        pts = class_points[c]
        for p in pts:
            X_res_list.append(p)
            y_res_list.append(c)

    X_res = np.vstack(X_res_list) if len(X_res_list) > 0 else np.empty((0, X.shape[1]))
    y_res = np.array(y_res_list, dtype=y.dtype)
    return X_res, y_res

In [ ]:
classifiers = {
    "SVC": SVC(probability=False, random_state=42),
    "RF": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(),
    "DecisionTree": DecisionTreeClassifier(),
    "RandomForest": RandomForestClassifier(),
    "MLPClassifier": MLPClassifier(),
    "EasyEnsembleClassifier": EasyEnsembleClassifier(),
    "RUSBoostClassifier": RUSBoostClassifier(),
    "BalancedBaggingClassifier": BalancedBaggingClassifier(),
    "BalancedRandomForestClassifier": BalancedRandomForestClassifier()
}

def infer_label_column(df):
    for cand in LABEL_COLUMNS:
        if cand in df.columns:
            return cand
    # return last column
    return df.columns[-1]

# The wrapper method to return resamplers to accept `k_neighbors` or `n_neighbors` as kwargs.
def resamplers_factory():
    # LASO placeholder with radius
    def laso_r_fn(X, y, k_neighbors=5, **kw):
        return laso_r(X, y, target_per_class=None, random_state=42,
                      k_neighbors=k_neighbors, var_explained=0.95,
                      max_components=None, max_retries=50, perturb_scale=1e-6)

    # LASO placeholder without radius
    def laso_fn(X, y, k_neighbors=5, **kw):
        return laso(X, y, target_per_class=None, random_state=42,
                       k_neighbors=k_neighbors, var_explained=0.95,
                       max_components=None, max_retries=50, perturb_scale=1e-6)
    
    def ro_fn(X, y, k_neighbors=5, **kw):
        ro = RandomOverSampler(random_state=42)
        return ro.fit_resample(X, y)

    def ru_fn(X, y, k_neighbors=5, **kw):
        ru = RandomUnderSampler(random_state=42)
        return ru.fit_resample(X, y)

    def nm_fn(X, y, k_neighbors=5, **kw):
        nm = NearMiss()
        return nm.fit_resample(X, y)

    def adasyn_fn(X, y, k_neighbors=5, **kw):
        a = ADASYN(n_neighbors=k_neighbors, random_state=42)
        return a.fit_resample(X, y)

    def blsmote_fn(X, y, k_neighbors=5, **kw):
        bl = BorderlineSMOTE(k_neighbors=k_neighbors, random_state=42)
        return bl.fit_resample(X, y)

    def smote_fn(X, y, k_neighbors=5, **kw):
        sm = SMOTE(k_neighbors=k_neighbors, random_state=42)
        return sm.fit_resample(X, y)

    def smoteenn_fn(X, y, k_neighbors=5, **kw):
        se = SMOTEENN(smote=SMOTE(k_neighbors=k_neighbors, random_state=42), random_state=42)
        Xs, ys = se.fit_resample(X, y)
        return Xs, ys

    def smotetomek_fn(X, y, k_neighbors=5, **kw):
        st = SMOTETomek(smote=SMOTE(k_neighbors=k_neighbors, random_state=42), random_state=42)
        Xs, ys = st.fit_resample(X, y)
        return Xs, ys

    def enn_fn(X, y, n_neighbors=3, **kw):
        enn = EditedNearestNeighbours(n_neighbors=n_neighbors)
        return enn.fit_resample(X, y)
    
    def tomeklinks_fn(X, y, k_neighbors=5, **kw):
        tl = TomekLinks()
        Xs, ys = tl.fit_resample(X, y)
        return Xs, ys

    def editedcdnn_fn(X, y, k_neighbors=5, **kw):
        edn = EditedCDNN(n_neighbors=k_neighbors)
        Xs, ys = edn.fit_resample(X, y)
        return Xs, ys

    def smotecdnn_fn(X, y, k_neighbors=5, **kw):
        scdnn = SMOTECDNN(smote=SMOTE(k_neighbors=k_neighbors, random_state=42), random_state=42)
        Xs, ys = scdnn.fit_resample(X, y)
        return Xs, ys

    def svmsmote_fn(X, y, k_neighbors=5, **kw):
        svmsmote = SVMSMOTE(k_neighbors=k_neighbors, random_state=42)
        Xs, ys = svmsmote.fit_resample(X, y)
        return Xs, ys

    def kmeanssmote_fn(X, y, k_neighbors=5, **kw):
        kmeanssmote = KMeansSMOTE(k_neighbors=k_neighbors, random_state=42)
        Xs, ys = kmeanssmote.fit_resample(X, y)
        return Xs, ys

    res = {
        "RO": ro_fn,
        "RU": ru_fn,
        "NearMiss": nm_fn,
        "ADASYN": adasyn_fn,
        "BL-SMOTE": blsmote_fn,
        "SMOTE": smote_fn,
        "SMOTEENN": smoteenn_fn,
        "SMOTETomek": smotetomek_fn,
        "ENN": enn_fn,
        "TomekLinks": tomeklinks_fn,
        "EditedCDNN": editedcdnn_fn,
        "SMOTECDNN": smotecdnn_fn,
        "SVMSMOTE": svmsmote_fn,
        "KMeansSMOTE": kmeanssmote_fn,
        #"LASO-R": laso_r_fn,
        "LASO": laso_fn
    }
    return res

# perform resampling while reducing neighbor when failure
def try_resample_with_decreasing_k(resampler_fn, X, y, start_k=5, min_k=1, param_name_candidates=('k_neighbors','n_neighbors','k'), **kwargs):
    last_exc = None
    for k in range(start_k, min_k - 1, -1):
        call_kwargs = dict(kwargs)
        for pname in param_name_candidates:
            call_kwargs[pname] = k
        try:
            result = resampler_fn(X, y, **call_kwargs)
            if isinstance(result, tuple) and len(result) == 2:
                Xr, yr = result
                if Xr is None or yr is None or len(yr) == 0:
                    raise RuntimeError(f"Resampler returned empty result for k={k}")
                return Xr, yr, k
            else:
                raise RuntimeError("Resampler returned unexpected object (expect (X_res,y_res)).")
        except Exception as e:
            last_exc = e
            # continue try lower k
            continue
    # if reached here, it means all k failed, raise last exception
    raise last_exc

# Friedman helper functions
def compute_ranks_and_friedman(results_df, methods):
    data = results_df[methods].values
    nan_mask = np.isnan(data)
    data_for_rank = data.copy()
    data_for_rank[nan_mask] = -1e9
    # Ranks: rank 1 = best; need to average ranks across datasets
    # Use rankdata with 'average' on negative so highest score -> rank 1
    ranks = np.apply_along_axis(lambda row: stats.rankdata(-row, method='average'), axis=1, arr=data_for_rank)
    ranks_df = pd.DataFrame(ranks, index=results_df.index, columns=methods)
    avg_ranks = ranks_df.mean(axis=0)
    arrays_for_friedman = [results_df[m].fillna(-1e9).values for m in methods]
    try:
        friedman_stat, friedman_p = stats.friedmanchisquare(*arrays_for_friedman)
    except Exception:
        friedman_stat, friedman_p = np.nan, np.nan
    return avg_ranks, friedman_stat, friedman_p, ranks_df

# Nemenyi helper functions
def nemenyi_posthoc(avg_ranks, N):
    from scipy.stats import norm
    methods = list(avg_ranks.index)
    K = len(methods)
    SE = math.sqrt(K * (K + 1) / (6.0 * N))
    pvals = pd.DataFrame(np.ones((K, K)), index=methods, columns=methods)
    zvals = pd.DataFrame(np.zeros((K, K)), index=methods, columns=methods)
    for i, mi in enumerate(methods):
        for j, mj in enumerate(methods):
            if i == j:
                pvals.loc[mi, mj] = 1.0
                zvals.loc[mi, mj] = 0.0
            else:
                z = abs(avg_ranks[mi] - avg_ranks[mj]) / SE
                p = 2.0 * (1.0 - norm.cdf(z))
                pvals.loc[mi, mj] = p
                zvals.loc[mi, mj] = z
    # significance boolean matrix
    sig = pvals < 0.05
    return pvals, zvals, sig, SE

# Helper function to plot resampling results
def plot_resampling_grid(
    X_original,
    y_original,
    resampled_dict,
    dataset_name,
    output_dir,
    use_tsne=0,                 # (1 = use t-SNE)
    tsne_perplexity=30,
    random_state=42
):
    # Encode labels once
    le = LabelEncoder()
    y_original_enc = le.fit_transform(y_original)

    all_items = [("Original", (X_original, y_original_enc))]

    # Encode all resampled labels consistently
    for mname, (Xr, yr) in resampled_dict.items():
        if Xr is None:
            all_items.append((mname, (Xr, yr)))
            continue            
        try:
            yr_enc = le.transform(yr)
        except:
            yr_enc = LabelEncoder().fit_transform(yr)
        all_items.append((mname, (Xr, yr_enc)))

    n_plots = len(all_items)
    n_cols = 3
    n_rows = math.ceil(n_plots / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten()

    for idx, (title, (X, y)) in enumerate(all_items):
        ax = axes[idx]

        # Error placeholder
        if X is None:
    
            # Keep axes visible
            ax.set_title(title)
            ax.set_xlabel("Feature 1" if use_tsne == 0 else "t-SNE 1")
            ax.set_ylabel("Feature 2" if use_tsne == 0 else "t-SNE 2")
    
            # Optional fixed limits so blank plots look consistent
            ax.set_xlim(-1, 1)
            ax.set_ylim(-1, 1)
    
            # Draw ERROR annotation
            ax.text(
                0.5,                  # center x (axes coords)
                0.5,                  # center y
                "resampling\nfailed",
                fontsize=20,
                color='red',
                ha='center',
                va='center',
                rotation=30,
                alpha=0.7,
                transform=ax.transAxes
            )
    
            # Skip normal plotting
            continue

        # Projection logic
        if use_tsne == 1:
            # t-SNE projection
            tsne = TSNE(n_components=2, perplexity=tsne_perplexity, random_state=random_state)
            X_plot = tsne.fit_transform(X)
            xlabel, ylabel = "t-SNE 1", "t-SNE 2"
        else:
            # Simple scatter (first 2 features)
            X_plot = X[:, :2] if X.shape[1] > 2 else X
            xlabel, ylabel = "Feature 1", "Feature 2"

        #print(title, X.shape)
        ax.scatter(X_plot[:, 0], X_plot[:, 1], c=y, s=10, cmap='tab10')
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)

    # Hide unused axes
    for j in range(idx + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    mode = "tsne" if use_tsne == 1 else "scatter"
    save_path = os.path.join(output_dir, f"{dataset_name}_{mode}_grid.png")

    plt.savefig(save_path, dpi=300)
    plt.clf()        # Clear the current figure
    plt.cla()        # Clear the current axes
    plt.close(fig)   # Close the specific figure object
    plt.close('all') # Force close any lingering hidden figures

    print(f"[Saved] {mode.upper()} grid: {save_path}")

# Helper function to compute AUC score
def compute_auc(results, model, X_test, y_test, y_pred):
    try:
        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(X_test)

            classes = np.unique(y_test)
            y_test_bin = label_binarize(y_test, classes=classes)

            if y_score.shape[1] == 2:
                # binary case
                results['auc'] = roc_auc_score(y_test, y_score[:, 1])
            else:
                # multiclass
                results['auc'] = roc_auc_score(
                    y_test_bin,
                    y_score,
                    average='macro',
                    multi_class='ovr'
                )
        else:
            results['auc'] = np.nan
    except Exception as e:
        print("AUC error:", e)
        results['auc'] = np.nan

# Helper function to compute classification metrics
def compute_metrics(model, X_test, y_test, y_pred):
    results = {}

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro')
    results['accuracy'] = float(accuracy)
    results['precision'] = float(precision)
    results['recall'] = float(recall)
    results['f1'] = float(f1)
        
    if 'gmean_score' in globals():
        results['gmean'] = gmean_score(y_test, y_pred)
    else:
        from sklearn.metrics import balanced_accuracy_score
        results['gmean'] = balanced_accuracy_score(y_test, y_pred)

    # --- AUC-ROC ---    
    # compute_auc(results, model, X_test, y_test, y_pred)

    return results

# Helper function to plot AUC results
def plot_auc_grid(models_dict, X_test, y_test, dataset_name, output_dir):

    n_plots = len(models_dict)
    n_cols = 3
    n_rows = int(np.ceil(n_plots / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten()

    for idx, (name, model) in enumerate(models_dict.items()):
        ax = axes[idx]

        try:
            if not hasattr(model, "predict_proba"):
                ax.set_title(f"{name} (no proba)")
                continue

            y_score = model.predict_proba(X_test)

            if y_score.shape[1] == 2:
                fpr, tpr, _ = roc_curve(y_test, y_score[:, 1])
                roc_auc = auc(fpr, tpr)

                ax.plot(fpr, tpr)
                ax.set_title(f"{name} (AUC={roc_auc:.3f})")
                ax.set_xlabel("FPR")
                ax.set_ylabel("TPR")

            else:
                ax.set_title(f"{name} (multiclass)")
        except Exception as e:
            ax.set_title(f"{name} error")

    for j in range(idx + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)

    save_path = os.path.join(output_dir, f"{dataset_name}_auc_grid.png")
    plt.savefig(save_path, dpi=300)

    plt.close(fig)
    plt.close('all')

    print(f"[Saved] AUC grid: {save_path}")

# Helper function to fill NaN values
def fill_nan(metrics_tables, ds_name, mname):
    for metric_name in metrics_tables.keys():
        if not pd.isna(metrics_tables[metric_name].loc[ds_name, mname]):
            continue
        metrics_tables[metric_name].loc[ds_name, mname] = np.nan

# Helper function to compute Dolan More Profile
def plot_dolan_more_profile(
    runtime_df,
    title,
    output_path,
    xlabel="Performance Ratio"
):
    plt.figure(figsize=(8, 6))
    # =====================================================
    # Compute performance ratios
    # =====================================================
    ratios = runtime_df.copy().astype(float)
    for ds in ratios.index:
        row = ratios.loc[ds]
        # successful methods only
        valid = row.dropna()
        if len(valid) == 0:
            continue

        best = valid.min()
        # avoid divide-by-zero
        best = max(best, 1e-12)
        ratios.loc[ds, valid.index] = valid / best

    # =====================================================
    # Build tau range using FINITE values only
    # =====================================================
    finite_vals = ratios.values[np.isfinite(ratios.values)]

    if len(finite_vals) == 0:
        print("No finite runtime ratios available.")
        return

    tau_max = finite_vals.max() + 0.5
    taus = np.linspace(1, tau_max, 1000)
    # =====================================================
    # Build Dolan-More curves
    # =====================================================
    for method in ratios.columns:
        method_ratios = ratios[method].values.astype(float)
        # -------------------------------------------------
        # Penalize failures permanently
        # -------------------------------------------------
        method_ratios = np.where(np.isnan(method_ratios), np.inf, method_ratios)

        rho = []
        for tau in taus:
            frac = np.mean(method_ratios <= tau)
            rho.append(frac)
            
        plt.plot(taus, rho, label=method)
    # =====================================================
    # Plot formatting
    # =====================================================
    plt.xlabel(r'$\tau$')
    plt.ylabel(r'$\rho(\tau)$')
    plt.title(title)
    plt.xlim(left=1)
    plt.xscale('log')
    ax = plt.gca()
    ax.minorticks_off()
    ax.grid(True, which='major', alpha=0.3)
    plt.ylim(0, 1.02)    
    plt.legend(fontsize=8, loc='lower right', ncol=2)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()
    print(f"[Saved] Dolan-More profile: {output_path}")

# Helper function to append global variables for Dolan More profiling
def append_global_runtime(
    container,
    ds_name,
    clf_name,
    mname,
    value
):
    """
    Append runtime result (success or failure) for Dolan-More profiling.
    """
    problem_id = f"{ds_name}__{clf_name}"
    container.append({'problem': problem_id,'method': mname,'value': value})

# Helper function to record benchmark failures
def append_failure_log(
    failure_container,
    ds_name,
    clf_name,
    mname,
    stage,
    exception_obj
):
    failure_container.append({'dataset': ds_name, 'classifier': clf_name, 'method': mname, 'stage': stage, 
                              'error_type': type(exception_obj).__name__, 'error_message': str(exception_obj)})

# loop classifiers and datasets
def run_pipeline(data_dir=DATA_DIR, classifiers_dict=classifiers,
                 resampler_factory=resamplers_factory, output_dir=OUTPUT_DIR,
                 start_k=5, min_k=1, test_split_fn=None):
    # find dataset files
    files = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
    if len(files) == 0:
        raise RuntimeError(f"No CSV files found in {data_dir}")

    # Determine resamplers
    resamplers = resampler_factory()
    methods = list(resamplers.keys())

    # test split function
    if test_split_fn is None:
        if 'stratified_small_test_split' in globals():
            test_split_fn = stratified_small_test_split
        else:
            from sklearn.model_selection import train_test_split
            def test_split_fn(X, y):
                return train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)

    global_resample_runtime = []
    global_pipeline_runtime = []
    global_failure_log = []

    # iterate classifiers
    for clf_name, clf_inst in classifiers_dict.items():
        print(f"CLASSIFIER: {clf_name} ======================")
        datasets = [os.path.basename(f) for f in files]
        # DataFrame to collect results (rows=datasets, cols=methods)
        metrics_tables = {
            'accuracy': pd.DataFrame(index=[os.path.basename(f) for f in files], columns=methods, dtype=float),
            'precision': pd.DataFrame(index=[os.path.basename(f) for f in files], columns=methods, dtype=float),
            'recall': pd.DataFrame(index=[os.path.basename(f) for f in files], columns=methods, dtype=float),
            'f1': pd.DataFrame(index=[os.path.basename(f) for f in files], columns=methods, dtype=float),
            'gmean': pd.DataFrame(index=[os.path.basename(f) for f in files], columns=methods, dtype=float),
            #'auc': pd.DataFrame(index=datasets, columns=methods, dtype=float),
            'resample_time': pd.DataFrame(index=[os.path.basename(f) for f in files], columns=methods, dtype=float),        
            'train_time': pd.DataFrame(index=[os.path.basename(f) for f in files], columns=methods, dtype=float),
            'predict_time': pd.DataFrame(index=[os.path.basename(f) for f in files], columns=methods, dtype=float),
            'pipeline_time': pd.DataFrame(index=[os.path.basename(f) for f in files], columns=methods, dtype=float),
        }
        classifier_failure_log = []

        for fidx, fpath in enumerate(files):
            resampled_storage = {}
            models_storage = {}
            ds_name = os.path.basename(fpath)
            print(f"DATASET {ds_name}: ----------------------") 
            try:
                df = pd.read_csv(fpath, sep=DELIM)                
            except Exception as e:
                print(f"Could not read {fpath}: {e}")
                continue

            label_col = infer_label_column(df)
            print("Inferred label column:", label_col)
            df = df.dropna()
            X_df = df.drop(columns=[label_col])
            y = df[label_col].values
        
            # Convert non-numeric columns using get_dummies
            non_numeric = X_df.select_dtypes(include=['object', 'category']).columns.tolist()
            if len(non_numeric)>0:
                X_df = pd.get_dummies(X_df, columns=non_numeric, drop_first=True)
                print("One-hot encoded columns:", non_numeric)
            # Ensure no remaining non-numeric
            for c in X_df.columns:
                if X_df[c].dtype == 'object':
                    X_df[c] = pd.to_numeric(X_df[c], errors='coerce').fillna(0)
            X = X_df.values.astype(float)

            # Preprocess numeric / categorical -> assume previous notebook handled this; here scale numeric
            scaler = StandardScaler()
            X = scaler.fit_transform(X)

            # split
            try:
                X_train, X_test, y_train, y_test = test_split_fn(X, y)
            except Exception as e:
                print("Split exception:", e)
                # fallback simple stratified split
                from sklearn.model_selection import train_test_split
                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)

            # for each method try resampling (with decreasing k if needed) and evaluate
            for mname, mfn in resamplers.items():      
                print(f"RESAMPLING: {mname}")
                try:
                    # attempt to resample with neighbor reduction wrapper
                    start_time = time.perf_counter()
                    Xr, yr, used_k = try_resample_with_decreasing_k(mfn, X_train, y_train, start_k=start_k, min_k=min_k)
                    resample_time = time.perf_counter() - start_time
                    metrics_tables["resample_time"].loc[ds_name, mname] = resample_time
                except Exception as e:
                    # mark ALL metrics as NaN
                    fill_nan(metrics_tables, ds_name, mname)
                    print(f"Resampling ERROR: failed on all k (error: {e})")
                    append_failure_log(classifier_failure_log, ds_name, clf_name, mname, stage='resampling', exception_obj=e)                    
                    append_failure_log(global_failure_log, ds_name, clf_name, mname, stage='resampling', exception_obj=e)
                    resampled_storage[mname] = (None, None)
                    append_global_runtime(global_resample_runtime, ds_name, clf_name, mname, np.nan)                    
                    append_global_runtime(global_pipeline_runtime, ds_name, clf_name, mname, np.nan)
                    continue

                resampled_storage[mname] = (Xr, yr)

                # Train classifier and evaluate                
                try:
                    start_time = time.perf_counter()
                    model = clone(clf_inst)
                    model.fit(Xr, yr)
                    train_time = time.perf_counter() - start_time
                    models_storage[mname] = model
                    predict_start = time.perf_counter()
                    y_pred = model.predict(X_test)
                    predict_time = time.perf_counter() - predict_start
                    metrics = compute_metrics(model, X_test, y_test, y_pred)
                    metrics['resample_time'] = resample_time
                    metrics['train_time'] = train_time
                    metrics['predict_time'] = predict_time
                    metrics['pipeline_time'] = (
                        resample_time +
                        train_time +
                        predict_time
                    )
                    append_global_runtime(global_resample_runtime, ds_name, clf_name, mname, metrics['resample_time'])                    
                    append_global_runtime(global_pipeline_runtime, ds_name, clf_name, mname, metrics['pipeline_time'])
                    for key in metrics_tables.keys():
                        metrics_tables[key].loc[ds_name, mname] = metrics[key]                    
                except Exception as e:
                    print("Predict exception:", e)                    
                    fill_nan(metrics_tables, ds_name, mname)
                    print(f"Classifier ERROR: failed after resampling by {mname}: {e}")
                    append_failure_log(classifier_failure_log, ds_name, clf_name, mname, stage='classification', exception_obj=e)
                    append_failure_log(global_failure_log, ds_name, clf_name, mname, stage='classification', exception_obj=e)
                    append_global_runtime(global_resample_runtime, ds_name, clf_name, mname, np.nan)
                    append_global_runtime(global_pipeline_runtime, ds_name, clf_name, mname, np.nan)

            # After all resampling methods for this dataset
            try:
                plot_resampling_grid(
                    X_train,
                    y_train,
                    resampled_storage,
                    dataset_name=ds_name.replace(".csv", ""),
                    output_dir=os.path.join(output_dir, clf_name),
                    use_tsne=0
                )
                """
                plot_auc_grid(
                    models_storage,
                    X_test,
                    y_test,
                    dataset_name=ds_name.replace(".csv", ""),
                    output_dir=os.path.join(output_dir, clf_name)
                )
                """
            except Exception as e:
                print(f"Plotting ERROR: failed for {ds_name}: {e}")

        # ---------------------------------------------------------
        # Keep only valid methods
        # ---------------------------------------------------------  
        results_table = metrics_tables['gmean']
        valid_methods = [m for m in methods if not results_table[m].isna().all()]
        if len(valid_methods) < 2:
            print(f"Not enough valid methods for classifier {clf_name} to perform Friedman test.")
            out_path = os.path.join(output_dir, f"{clf_name}.xlsx")
            results_table.to_excel(out_path, sheet_name="raw_scores")
            continue
            
        sub_df = results_table[valid_methods].copy()        
        sub_df = sub_df.dropna(how='all', subset=valid_methods)
        
        N = sub_df.shape[0]
        # ---------------------------------------------------------
        # Friedman test + ranks
        # ---------------------------------------------------------
        avg_ranks, friedman_stat, friedman_p, ranks_df = (compute_ranks_and_friedman(sub_df, valid_methods))
        
        # ---------------------------------------------------------
        # Nemenyi posthoc
        # ---------------------------------------------------------        
        pvals_df, zvals_df, sig_df, SE = (nemenyi_posthoc(avg_ranks, N))

        # Save and include raw scores, ranks, avg ranks, friedman summary, pvals, significance
        median_resample_time = metrics_tables['resample_time'].apply(np.nanmedian, axis=0)
        median_pipeline_time = (
            metrics_tables['resample_time'] +
            metrics_tables['train_time'] +
            metrics_tables['predict_time']
        ).apply(np.nanmedian, axis=0)
        std_resample_time = metrics_tables['resample_time'].std(axis=0)
        std_pipeline_time = metrics_tables['pipeline_time'].std(axis=0)
        runtime_summary_df = pd.DataFrame({
            'median_resample_time': median_resample_time,
            'std_resample_time': std_resample_time,
            'median_pipeline_time': median_pipeline_time,
            'std_pipeline_time': std_pipeline_time
        })

        summary_df = pd.DataFrame({
            'friedman_stat': [friedman_stat],
            'friedman_pval': [friedman_p],
            'N_datasets': [N],
            'K_methods': [len(valid_methods)],
            'SE_nemenyi': [SE]
        })

        classifier_failure_df = pd.DataFrame(classifier_failure_log)
        
        out_path = os.path.join(output_dir, f"{clf_name}.xlsx")
        with pd.ExcelWriter(out_path, engine='xlsxwriter') as writer:
            runtime_summary_df.to_excel(writer, sheet_name='runtime_summary')
            summary_df.to_excel(writer, sheet_name='friedman_summary', index=False)
            sub_df.to_excel(writer, sheet_name='friedman_scores')
            ranks_df.to_excel(writer, sheet_name='friedman_ranks')
            avg_ranks.to_frame('avg_rank').to_excel(writer, sheet_name='avg_ranks')
            pvals_df.to_excel(writer, sheet_name='nemenyi_pvals')
            zvals_df.to_excel(writer, sheet_name='nemenyi_zvals')
            sig_df.astype(int).to_excel(writer, sheet_name='nemenyi_significant')
            for metric_name, df_metric in metrics_tables.items():
                df_metric.to_excel(writer, sheet_name=metric_name)
            classifier_failure_df.to_excel(writer, sheet_name='failure_log', index=False)
        
        print(f"Output classifier {clf_name} to {out_path}")

        plot_dolan_more_profile(
            metrics_tables['resample_time'],
            title=f"{clf_name} - Resampling Runtime Profile",
            output_path=os.path.join(
                output_dir,
                f"{clf_name}_resampling_profile.png"
            )
        )

        plot_dolan_more_profile(
            metrics_tables['pipeline_time'],
            title=f"{clf_name} - Pipeline Runtime Profile",
            output_path=os.path.join(
                output_dir,
                f"{clf_name}_pipeline_profile.png"
            )
        )

    global_failure_df = pd.DataFrame(global_failure_log)
    failure_summary_df = (global_failure_df.groupby(['method']).size().reset_index(name='failure_count')
        .sort_values(by='failure_count',ascending=False))
    failure_detail_df = (global_failure_df.groupby(['method', 'dataset', 'classifier']).size().reset_index(name='count'))
    # Total benchmark tasks
    total_tasks = (len(files) * len(classifiers_dict))        
    # Failure count per method
    failure_rate_df = pd.DataFrame({'method': methods})
    failure_counts = (global_failure_df.groupby('method').size())
    failure_rate_df['failure_count'] = (failure_rate_df['method'].map(failure_counts).fillna(0).astype(int))
    
    # Convert to rate
    failure_rate_df['failure_rate'] = (failure_rate_df['failure_count']/ total_tasks)
    # Success rate
    failure_rate_df['success_rate'] = (1.0 - failure_rate_df['failure_rate'])
    # Sort by failure rate
    failure_rate_df = failure_rate_df.sort_values(by='failure_rate', ascending=False)
    failure_heatmap_df = pd.pivot_table(global_failure_df, index='method', columns='classifier', aggfunc='size', fill_value=0)
    failure_heatmap_df = failure_heatmap_df.reindex(methods, fill_value=0)
    dataset_failure_heatmap_df = pd.pivot_table(global_failure_df, index='method', columns='dataset', aggfunc='size', fill_value=0)
    dataset_failure_heatmap_df = dataset_failure_heatmap_df.reindex(methods, fill_value=0)
        
    global_failure_path = os.path.join(output_dir, "GLOBAL_failure_analysis.xlsx")
    global_stage_failure_heatmap_df = pd.pivot_table(global_failure_df, index='method', columns='stage', aggfunc='size', fill_value=0)
    global_stage_failure_heatmap_df = (global_stage_failure_heatmap_df.reindex(methods, fill_value=0))
    with pd.ExcelWriter(global_failure_path,engine='xlsxwriter') as writer:
        failure_rate_df.to_excel(writer, sheet_name='failure_rate', index=False)         
        failure_heatmap_df.to_excel(writer, sheet_name='failure_heatmap')     
        dataset_failure_heatmap_df.to_excel(writer, sheet_name='dataset_failures')
        global_stage_failure_heatmap_df.to_excel(writer, sheet_name='classifier_stage_failures')
        global_failure_df.to_excel(writer, sheet_name='all_failures', index=False)      
        failure_summary_df.to_excel(writer, sheet_name='failure_summary', index=False)
        failure_detail_df.to_excel(writer, sheet_name='failure_details', index=False)

    global_resample_df = pd.DataFrame(global_resample_runtime)
    global_pipeline_df = pd.DataFrame(global_pipeline_runtime)
    global_resample_df = global_resample_df.pivot_table(index='problem', columns='method', values='value', aggfunc='first')    
    global_pipeline_df = global_pipeline_df.pivot_table(index='problem', columns='method', values='value', aggfunc='first')
    plot_dolan_more_profile(
        global_resample_df,
        title="Global Resampling Runtime Profile",
        output_path=os.path.join(
            output_dir,
            "GLOBAL_resampling_profile.png"
        )
    )
    
    plot_dolan_more_profile(
        global_pipeline_df,
        title="Global Pipeline Runtime Profile",
        output_path=os.path.join(
            output_dir,
            "GLOBAL_pipeline_profile.png"
        )
    )

In [ ]:
# =========================================================
# Pipeline execution timing
# =========================================================
pipeline_start_datetime = datetime.now()
pipeline_start_perf = time.perf_counter()

print("=" * 60)
print(f"PIPELINE START : {pipeline_start_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

# =========================================================
# RUN PIPELINE
# =========================================================
run_pipeline()

# =========================================================
# Pipeline execution timing end
# =========================================================
pipeline_end_datetime = datetime.now()
pipeline_elapsed_seconds = time.perf_counter() - pipeline_start_perf

hours = int(pipeline_elapsed_seconds // 3600)
minutes = int((pipeline_elapsed_seconds % 3600) // 60)
seconds = pipeline_elapsed_seconds % 60

print("=" * 60)
print(f"PIPELINE END   : {pipeline_end_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
print(
    f"TOTAL DURATION : "
    f"{hours:02d}h "
    f"{minutes:02d}m "
    f"{seconds:06.3f}s"
)
print("=" * 60)